# 01 — Exploratory Data Analysis: Steam 200K Dataset

Understanding the raw data before building any pipeline. This notebook covers:
- Dataset shape, columns, and data types
- Play vs purchase behavior breakdown
- Playtime distribution (log scale)
- Interactions per user and per game
- Sparsity of the user-item matrix
- Top 20 most played games

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Load Raw Data

In [ ]:
RAW_PATH = '../data/raw/steam-200k.csv'

df_raw = pd.read_csv(
    RAW_PATH,
    header=None,
    names=['user_id', 'game_name', 'behavior', 'value', 'drop']
)
df_raw = df_raw.drop(columns=['drop'])

print(f'Shape: {df_raw.shape}')
print(f'Dtypes:\n{df_raw.dtypes}')
print(f'\nNull counts:\n{df_raw.isnull().sum()}')
df_raw.head(10)

In [ ]:
# Behavior breakdown
print('Behavior counts:')
print(df_raw['behavior'].value_counts())
print(f'\nUnique users (raw): {df_raw["user_id"].nunique():,}')
print(f'Unique games (raw): {df_raw["game_name"].nunique():,}')

## 2. Filter to Play-Only Rows

Only `play` rows contain playtime signal. `purchase` rows add no information we can't infer from play.

In [ ]:
df = df_raw[df_raw['behavior'] == 'play'].copy()
df = df.rename(columns={'value': 'playtime'})
df['playtime'] = df['playtime'].astype(float)

# Drop zero-playtime rows (can't be a true interaction)
df = df[df['playtime'] > 0].reset_index(drop=True)

print(f'Play-only rows: {len(df):,}')
print(f'Unique users:   {df["user_id"].nunique():,}')
print(f'Unique games:   {df["game_name"].nunique():,}')
df.head()

## 3. Playtime Distribution

Playtime is heavily right-skewed. Log scale reveals the full shape.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Linear scale
axes[0].hist(df['playtime'].clip(upper=500), bins=60, color='steelblue', edgecolor='none')
axes[0].set_title('Playtime distribution (clipped at 500h)')
axes[0].set_xlabel('Playtime (hours)')
axes[0].set_ylabel('Count')

# Log scale
axes[1].hist(np.log1p(df['playtime']), bins=60, color='coral', edgecolor='none')
axes[1].set_title('Playtime distribution (log scale)')
axes[1].set_xlabel('log(1 + playtime)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/processed/eda_playtime_dist.png', bbox_inches='tight')
plt.show()

print(df['playtime'].describe().round(2))

## 4. Interactions per User

How many games has each user played? Most users have very few interactions — this is the long tail problem.

In [ ]:
user_counts = df.groupby('user_id')['game_name'].count().rename('num_games')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(user_counts.clip(upper=50), bins=50, color='steelblue', edgecolor='none')
axes[0].set_title('Games played per user (clipped at 50)')
axes[0].set_xlabel('Number of games played')
axes[0].set_ylabel('Number of users')

axes[1].hist(np.log1p(user_counts), bins=50, color='coral', edgecolor='none')
axes[1].set_title('Games played per user (log scale)')
axes[1].set_xlabel('log(1 + num_games)')
axes[1].set_ylabel('Number of users')

plt.tight_layout()
plt.savefig('../data/processed/eda_user_interactions.png', bbox_inches='tight')
plt.show()

print(user_counts.describe().round(2))
print(f'\nUsers with only 1 game played: {(user_counts == 1).sum():,} ({(user_counts == 1).mean()*100:.1f}%)')
print(f'Users with >= 5 games played:  {(user_counts >= 5).sum():,} ({(user_counts >= 5).mean()*100:.1f}%)')

## 5. Interactions per Game

How many users have played each game? Very few games dominate, most have very few players.

In [ ]:
game_counts = df.groupby('game_name')['user_id'].count().rename('num_users')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(game_counts.clip(upper=200), bins=60, color='steelblue', edgecolor='none')
axes[0].set_title('Users per game (clipped at 200)')
axes[0].set_xlabel('Number of players')
axes[0].set_ylabel('Number of games')

axes[1].hist(np.log1p(game_counts), bins=60, color='coral', edgecolor='none')
axes[1].set_title('Users per game (log scale)')
axes[1].set_xlabel('log(1 + num_players)')
axes[1].set_ylabel('Number of games')

plt.tight_layout()
plt.savefig('../data/processed/eda_game_interactions.png', bbox_inches='tight')
plt.show()

print(game_counts.describe().round(2))
print(f'\nGames with < 10 players: {(game_counts < 10).sum():,} ({(game_counts < 10).mean()*100:.1f}%)')
print(f'Games with >= 10 players: {(game_counts >= 10).sum():,} ({(game_counts >= 10).mean()*100:.1f}%)')

## 6. User-Item Matrix Sparsity

In [ ]:
n_users = df['user_id'].nunique()
n_games = df['game_name'].nunique()
n_interactions = len(df)
matrix_size = n_users * n_games
sparsity = 1.0 - n_interactions / matrix_size

print('=== User-Item Matrix Stats ===')
print(f'Unique users:       {n_users:,}')
print(f'Unique games:       {n_games:,}')
print(f'Total interactions: {n_interactions:,}')
print(f'Matrix size:        {matrix_size:,}')
print(f'Sparsity:           {sparsity*100:.4f}%')
print(f'Density:            {(1-sparsity)*100:.4f}%')
print(f'Avg games/user:     {n_interactions/n_users:.2f}')
print(f'Avg users/game:     {n_interactions/n_games:.2f}')

## 7. Top 20 Most Played Games

In [ ]:
top20_by_players = game_counts.nlargest(20).reset_index()
top20_by_players.columns = ['game_name', 'num_players']

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(
    top20_by_players['game_name'][::-1],
    top20_by_players['num_players'][::-1],
    color='steelblue'
)
ax.set_title('Top 20 Games by Number of Players')
ax.set_xlabel('Number of players in dataset')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('../data/processed/eda_top20_games.png', bbox_inches='tight')
plt.show()

## 8. Top 20 Games by Total Playtime

In [ ]:
top20_by_playtime = df.groupby('game_name')['playtime'].sum().nlargest(20).reset_index()
top20_by_playtime.columns = ['game_name', 'total_playtime_hours']

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(
    top20_by_playtime['game_name'][::-1],
    top20_by_playtime['total_playtime_hours'][::-1],
    color='coral'
)
ax.set_title('Top 20 Games by Total Playtime (hours)')
ax.set_xlabel('Total hours played across all users')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('../data/processed/eda_top20_playtime.png', bbox_inches='tight')
plt.show()

## 9. Impact of Filtering

The preprocessing pipeline drops users with < 5 interactions and items with < 10 interactions.
Let's see how much data survives.

In [ ]:
MIN_USER = 5
MIN_ITEM = 10

df_filt = df.copy()
for _ in range(5):
    uc = df_filt['user_id'].value_counts()
    gc = df_filt['game_name'].value_counts()
    df_filt = df_filt[
        df_filt['user_id'].isin(uc[uc >= MIN_USER].index) &
        df_filt['game_name'].isin(gc[gc >= MIN_ITEM].index)
    ]

print('=== After Filtering ===')
print(f'Rows:   {len(df):,} -> {len(df_filt):,} ({len(df_filt)/len(df)*100:.1f}% retained)')
print(f'Users:  {df["user_id"].nunique():,} -> {df_filt["user_id"].nunique():,}')
print(f'Games:  {df["game_name"].nunique():,} -> {df_filt["game_name"].nunique():,}')

n_u2 = df_filt['user_id'].nunique()
n_g2 = df_filt['game_name'].nunique()
sparsity2 = 1 - len(df_filt) / (n_u2 * n_g2)
print(f'Sparsity after filter: {sparsity2*100:.4f}%')

## 10. Summary

Key findings to carry into the preprocessing and modeling phases.

In [ ]:
print('=== EDA Summary ===')
print(f'Raw interactions (play only, >0h): {len(df):,}')
print(f'Raw users: {df["user_id"].nunique():,}  |  Raw games: {df["game_name"].nunique():,}')
print(f'Raw sparsity: {sparsity*100:.3f}%')
print()
print(f'After filtering (user>=5, item>=10):')
print(f'  Interactions: {len(df_filt):,}')
print(f'  Users: {df_filt["user_id"].nunique():,}  |  Games: {df_filt["game_name"].nunique():,}')
print(f'  Sparsity: {sparsity2*100:.3f}%')
print()
print(f'Median playtime per interaction: {df["playtime"].median():.1f}h')
print(f'Mean playtime per interaction:   {df["playtime"].mean():.1f}h')
print(f'Max playtime in dataset:         {df["playtime"].max():.1f}h')